# Analyse VLAD — Base MEB

Comparaison des représentations moyennées (`block_0`, `block_1`) et VLAD (K=16, K=32)
sur les patches annotés MEB — Linear Probing, Fisher Criterion, NNT Ratio, τ.
Protocole identique à `explore_database_meb.ipynb` (PCA-50d, 5-fold stratifié par image, CATS_EXCLUDE, cross-image).

In [ ]:
# ── Setup ──────────────────────────────────────────────────────────────────────
import h5py
import numpy as np
import json
import pickle
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score, classification_report
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
from scipy.stats import mode
from tqdm.auto import tqdm

ROOT     = Path('..').resolve()
DB_PATH  = ROOT / 'data' / 'feature_database' / 'database_meb.h5'
CFG_PATH = ROOT / 'PatchTagger_Output' / 'config' / 'config.json'
OUT_DIR  = ROOT / 'outputs' / 'vlad'
OUT_DIR.mkdir(parents=True, exist_ok=True)

SEED    = 42
PCA_DIM = 50
MIN_N   = 30
N_FOLDS = 5
C_REG   = 1.0

with open(CFG_PATH) as f:
    cfg = json.load(f)
CATEGORIES = {int(k): v['name'] for k, v in cfg['available_categories'].items()}

CATS_EXCLUDE = [2, 8, 10, 11, 12, 13]

with h5py.File(DB_PATH, 'r') as h5:
    IMAGE_NAMES  = h5['metadata/image_names'][:]
    CATEGORY_IDS = h5['metadata/category_ids'][:]
    POSITIONS    = h5['metadata/positions'][:]

CATS_VALID = sorted([
    c for c in np.unique(CATEGORY_IDS)
    if c not in CATS_EXCLUDE
    and (CATEGORY_IDS == c).sum() >= MIN_N
])
# → [1, 3, 4, 5, 6, 7, 9]

mask_valid = np.isin(CATEGORY_IDS, CATS_VALID)
y_valid    = CATEGORY_IDS[mask_valid]
imgs_valid = IMAGE_NAMES[mask_valid]

KEYS_MEAN = ['block_0', 'block_1']
KEYS_VLAD = [
    'block_0_vlad_k16', 'block_0_vlad_k32',
    'block_1_vlad_k16', 'block_1_vlad_k32',
]
KEYS_ALL = KEYS_MEAN + KEYS_VLAD

# K-Fold stratifié par image (catégorie dominante par image)
images_uniq = np.unique(IMAGE_NAMES)
cat_dom = np.array([
    int(mode(CATEGORY_IDS[IMAGE_NAMES == img]).mode)
    for img in images_uniq
])
skf   = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
FOLDS = list(skf.split(images_uniq, cat_dom))

print(f'Catégories valides : {[CATEGORIES[c] for c in CATS_VALID]}')
print(f'N patches valides  : {mask_valid.sum()}')
print(f'Clés à analyser    : {KEYS_ALL}')

## Cell 1 — Vérification VLAD dans la base

In [ ]:
with h5py.File(DB_PATH, 'r') as h5:
    print('=== CLÉS DISPONIBLES ===')
    for key in KEYS_ALL:
        if key in h5['features']:
            shape  = h5['features'][key].shape
            sample = h5['features'][key][:100]
            norms  = np.linalg.norm(sample, axis=1)
            tag    = 'VLAD' if 'vlad' in key else 'MEAN'
            print(f'  [{tag}] {key:<25} shape={shape}  norm={norms.mean():.4f} ✅')
        else:
            tag = 'VLAD' if 'vlad' in key else 'MEAN'
            print(f'  [{tag}] {key:<25} ❌ ABSENT  (lancer build_vlad_database.py)')

## Cell 2 — Linear Probing

In [ ]:
_vlad_lp_results = {}

with h5py.File(DB_PATH, 'r') as h5:
    for key in tqdm(KEYS_ALL, desc='Linear Probing'):
        if key not in h5['features']:
            print(f'  {key} ❌ ABSENT — ignoré')
            continue

        X_all        = h5['features'][key][:][mask_valid]
        fold_accs    = []
        fold_per_cat = {c: [] for c in CATS_VALID}
        fold_var_exp = []

        for tr_img_idx, te_img_idx in FOLDS:
            train_imgs = images_uniq[tr_img_idx]
            test_imgs  = images_uniq[te_img_idx]

            m_train = np.isin(imgs_valid, train_imgs)
            m_test  = np.isin(imgs_valid, test_imgs)

            X_train = X_all[m_train]
            y_train = y_valid[m_train]
            X_test  = X_all[m_test]
            y_test  = y_valid[m_test]

            # PCA-50d — fit sur train uniquement
            n_comp      = min(PCA_DIM, X_train.shape[1])
            pca         = PCA(n_components=n_comp, random_state=SEED)
            X_train_pca = pca.fit_transform(X_train)
            X_test_pca  = pca.transform(X_test)
            fold_var_exp.append(pca.explained_variance_ratio_.sum())

            # StandardScaler — fit sur train uniquement
            scaler     = StandardScaler()
            X_train_n  = scaler.fit_transform(X_train_pca)
            X_test_n   = scaler.transform(X_test_pca)

            clf = LogisticRegression(
                class_weight='balanced', max_iter=1000,
                C=C_REG, random_state=SEED,
                solver='lbfgs',
            )
            clf.fit(X_train_n, y_train)
            y_pred = clf.predict(X_test_n)

            fold_accs.append(balanced_accuracy_score(y_test, y_pred))

            report = classification_report(
                y_test, y_pred, labels=CATS_VALID,
                output_dict=True, zero_division=0,
            )
            for c in CATS_VALID:
                fold_per_cat[c].append(report.get(str(c), {}).get('recall', 0.0))

        _vlad_lp_results[key] = {
            'acc_mean'   : float(np.mean(fold_accs)),
            'acc_std'    : float(np.std(fold_accs)),
            'folds'      : fold_accs,
            'acc_per_cat': {c: float(np.mean(fold_per_cat[c])) for c in CATS_VALID},
            'var_pca'    : float(np.mean(fold_var_exp)),
        }

baseline   = 1.0 / len(CATS_VALID) * 100
acc_block0 = _vlad_lp_results.get('block_0', {}).get('acc_mean', 0.0) * 100

print(f'\nBaseline = 1/{len(CATS_VALID)} = {baseline:.1f}%\n')
print(f'{"Clé":<25} │ {"Acc":>10} │ {"Var PCA":>8} │ {"vs block_0":>10}')
print('─' * 65)
for key in [k for k in KEYS_ALL if k in _vlad_lp_results]:
    r     = _vlad_lp_results[key]
    acc   = r['acc_mean'] * 100
    std   = r['acc_std']  * 100
    var   = r['var_pca']  * 100
    delta = acc - acc_block0
    sign  = '+' if delta >= 0 else ''
    print(f'{key:<25} │ {acc:>5.1f}%±{std:>4.1f} │ {var:>7.1f}% │ {sign}{delta:>+.1f}%')

with open(OUT_DIR / 'lp_results.pkl', 'wb') as f:
    pickle.dump(_vlad_lp_results, f)
print(f'\n→ Résultats sauvegardés : {OUT_DIR}/lp_results.pkl')

## Cell 3 — Fisher Criterion balancé

In [ ]:
_vlad_fisher_results = {}

with h5py.File(DB_PATH, 'r') as h5:
    for key in tqdm(KEYS_ALL, desc='Fisher'):
        if key not in h5['features']:
            print(f'  {key} ❌ ABSENT — ignoré')
            continue

        X_raw = h5['features'][key][:][mask_valid]

        n_comp = min(PCA_DIM, X_raw.shape[1])
        pca    = PCA(n_components=n_comp, random_state=SEED)
        X_50   = pca.fit_transform(X_raw)

        N, D = X_50.shape
        mu   = X_50.mean(axis=0)

        S_B = np.zeros((D, D))
        S_W = np.zeros((D, D))

        for c in CATS_VALID:
            mask_c = y_valid == c
            N_c    = mask_c.sum()
            mu_c   = X_50[mask_c].mean(axis=0)
            diff   = (mu_c - mu).reshape(-1, 1)
            S_B   += diff @ diff.T                          # balancé (sans N_c)
            diff_c = X_50[mask_c] - mu_c
            S_W   += (1.0 / N_c) * (diff_c.T @ diff_c)     # balancé (1/N_c)

        J = float(np.trace(S_B) / (np.trace(S_W) + 1e-10))

        J_per_cat = {}
        for c in CATS_VALID:
            mu_c   = X_50[y_valid == c].mean(axis=0)
            diff   = (mu_c - mu).reshape(-1, 1)
            J_per_cat[c] = float(np.trace(diff @ diff.T) / (np.trace(S_W) + 1e-10))

        _vlad_fisher_results[key] = {'J': J, 'J_per_cat': J_per_cat}

J_block0 = _vlad_fisher_results.get('block_0', {}).get('J', 0.0)
print(f'\n{"Clé":<25} │ {"J balancé":>10} │ {"vs block_0":>10}')
print('─' * 50)
for key in [k for k in KEYS_ALL if k in _vlad_fisher_results]:
    J     = _vlad_fisher_results[key]['J']
    delta = J - J_block0
    sign  = '+' if delta >= 0 else ''
    print(f'{key:<25} │ {J:>10.4f} │ {sign}{delta:>+.4f}')

with open(OUT_DIR / 'fisher_results.pkl', 'wb') as f:
    pickle.dump(_vlad_fisher_results, f)
print(f'\n→ Résultats sauvegardés : {OUT_DIR}/fisher_results.pkl')

## Cell 4 — NNT ratio (cross-image)

In [ ]:
def _vlad_nnt_ratio(X_norm, cats, imgs, cat, k=5):
    """NNT ratio cross-image sur vecteurs L2-normalisés après PCA."""
    mask_c  = cats == cat
    imgs_c  = np.unique(imgs[mask_c])
    d_intra_all, d_inter_all = [], []

    for img_src in imgs_c:
        X_q     = X_norm[mask_c  & (imgs == img_src)]
        X_intra = X_norm[mask_c  & (imgs != img_src)]
        X_inter = X_norm[~mask_c & (imgs != img_src)]

        k_i = min(k, len(X_intra))
        k_e = min(k, len(X_inter))
        if k_i < 1 or k_e < 1:
            continue

        nbrs_i = NearestNeighbors(n_neighbors=k_i, algorithm='ball_tree', metric='euclidean').fit(X_intra)
        di, _  = nbrs_i.kneighbors(X_q)
        d_intra_all.append(di.mean())

        nbrs_e = NearestNeighbors(n_neighbors=k_e, algorithm='ball_tree', metric='euclidean').fit(X_inter)
        de, _  = nbrs_e.kneighbors(X_q)
        d_inter_all.append(de.mean())

    if not d_intra_all:
        return np.nan
    return np.mean(d_inter_all) / (np.mean(d_intra_all) + 1e-8)


_vlad_nnt_results = {}

with h5py.File(DB_PATH, 'r') as h5:
    for key in tqdm(KEYS_ALL, desc='NNT ratio'):
        if key not in h5['features']:
            print(f'  {key} ❌ ABSENT — ignoré')
            continue

        X_raw = h5['features'][key][:][mask_valid]

        n_comp = min(PCA_DIM, X_raw.shape[1])
        pca    = PCA(n_components=n_comp, random_state=SEED)
        X_pca  = pca.fit_transform(X_raw)
        norms  = np.linalg.norm(X_pca, axis=1, keepdims=True)
        X_norm = X_pca / np.where(norms < 1e-8, 1.0, norms)

        ratios = []
        for c in CATS_VALID:
            r = _vlad_nnt_ratio(X_norm, y_valid, imgs_valid, c, k=5)
            if not np.isnan(r):
                ratios.append(r)

        _vlad_nnt_results[key] = {
            'ratio_macro': float(np.mean(ratios)) if ratios else np.nan,
            'ratios'     : ratios,
        }

r_block0 = _vlad_nnt_results.get('block_0', {}).get('ratio_macro', 0.0)
print(f'\n{"Clé":<25} │ {"Ratio macro":>12} │ {"vs block_0":>10}')
print('─' * 53)
for key in [k for k in KEYS_ALL if k in _vlad_nnt_results]:
    r     = _vlad_nnt_results[key]['ratio_macro']
    delta = r - r_block0
    sign  = '+' if delta >= 0 else ''
    print(f'{key:<25} │ {r:>12.4f} │ {sign}{delta:>+.4f}')

with open(OUT_DIR / 'nnt_results.pkl', 'wb') as f:
    pickle.dump(_vlad_nnt_results, f)
print(f'\n→ Résultats sauvegardés : {OUT_DIR}/nnt_results.pkl')

## Cell 5 — τ cross / intra

In [ ]:
_vlad_tau_results = {}

with h5py.File(DB_PATH, 'r') as h5:
    for key in tqdm(KEYS_ALL, desc='Tau'):
        if key not in h5['features']:
            print(f'  {key} ❌ ABSENT — ignoré')
            continue

        X_raw  = h5['features'][key][:][mask_valid]

        n_comp = min(PCA_DIM, X_raw.shape[1])
        pca    = PCA(n_components=n_comp, random_state=SEED)
        X_50   = pca.fit_transform(X_raw)
        norms  = np.linalg.norm(X_50, axis=1, keepdims=True)
        X_norm = X_50 / np.where(norms < 1e-8, 1.0, norms)

        cross_per_cat = {}
        intra_per_cat = {}

        for c in CATS_VALID:
            mask_c = y_valid == c
            X_c    = X_norm[mask_c]
            imgs_c = imgs_valid[mask_c]
            N_c    = X_c.shape[0]

            sim    = X_c @ X_c.T
            upper  = np.triu(np.ones((N_c, N_c), dtype=bool), k=1)
            m_cross = (imgs_c[:, None] != imgs_c[None, :]) & upper
            m_intra = (imgs_c[:, None] == imgs_c[None, :]) & upper

            paires_cross = sim[m_cross]
            paires_intra = sim[m_intra]

            cross_per_cat[c] = float(paires_cross.mean()) if paires_cross.size > 0 else np.nan
            intra_per_cat[c] = float(paires_intra.mean()) if paires_intra.size > 0 else np.nan

        cross_vals = [v for v in cross_per_cat.values() if not np.isnan(v)]
        intra_vals = [v for v in intra_per_cat.values() if not np.isnan(v)]

        _vlad_tau_results[key] = {
            'cross_macro' : float(np.mean(cross_vals)),
            'intra_macro' : float(np.mean(intra_vals)),
            'ratio_macro' : float(np.mean(cross_vals)) / (float(np.mean(intra_vals)) + 1e-8),
            'cross'       : cross_per_cat,
            'intra'       : intra_per_cat,
        }

print(f'\n{"Clé":<25} │ {"τ cross":>8} │ {"τ intra":>8} │ {"ratio":>6}')
print('─' * 58)
for key in [k for k in KEYS_ALL if k in _vlad_tau_results]:
    r = _vlad_tau_results[key]
    print(f'{key:<25} │ {r["cross_macro"]:>8.3f} │ {r["intra_macro"]:>8.3f} │ {r["ratio_macro"]:>6.3f}')

with open(OUT_DIR / 'tau_results.pkl', 'wb') as f:
    pickle.dump(_vlad_tau_results, f)
print(f'\n→ Résultats sauvegardés : {OUT_DIR}/tau_results.pkl')

## Cell 6 — Figure synthèse comparative

In [ ]:
COLORS = {
    'block_0'          : '#1B4F72',
    'block_0_vlad_k16' : '#2E86AB',
    'block_0_vlad_k32' : '#A8DADC',
    'block_1'          : '#E76F51',
    'block_1_vlad_k16' : '#F4A261',
    'block_1_vlad_k32' : '#FFCBA4',
}

# Filtrer sur les clés effectivement calculées
keys_present = [
    k for k in KEYS_ALL
    if k in _vlad_lp_results
    and k in _vlad_fisher_results
    and k in _vlad_nnt_results
    and k in _vlad_tau_results
]

if not keys_present:
    print('Aucun résultat disponible — lancer les cells 2-5 d\'abord.')
else:
    x     = np.arange(len(keys_present))
    width = 0.6
    baseline = 1.0 / len(CATS_VALID) * 100

    fig, axes = plt.subplots(2, 2, figsize=(16, 12))

    # ── LP ────────────────────────────────────────────────────────────────────
    ax = axes[0, 0]
    vals = [_vlad_lp_results[k]['acc_mean'] * 100 for k in keys_present]
    errs = [_vlad_lp_results[k]['acc_std']  * 100 for k in keys_present]
    ax.bar(x, vals, width, color=[COLORS[k] for k in keys_present], yerr=errs, capsize=4)
    ax.axhline(baseline, color='red', ls=':', lw=1.5, label=f'Baseline {baseline:.1f}%')
    ax.set_xticks(x)
    ax.set_xticklabels(keys_present, rotation=45, ha='right', fontsize=8)
    ax.set_ylabel('Balanced Accuracy (%)')
    ax.set_title('Linear Probing — Moyenne vs VLAD')
    ax.legend(fontsize=8)

    # ── Fisher J ──────────────────────────────────────────────────────────────
    ax = axes[0, 1]
    vals = [_vlad_fisher_results[k]['J'] for k in keys_present]
    ax.bar(x, vals, width, color=[COLORS[k] for k in keys_present])
    ax.set_xticks(x)
    ax.set_xticklabels(keys_present, rotation=45, ha='right', fontsize=8)
    ax.set_ylabel('Fisher J balancé')
    ax.set_title('Fisher Criterion — Moyenne vs VLAD')

    # ── NNT ratio ─────────────────────────────────────────────────────────────
    ax = axes[1, 0]
    vals = [_vlad_nnt_results[k]['ratio_macro'] for k in keys_present]
    ax.bar(x, vals, width, color=[COLORS[k] for k in keys_present])
    ax.axhline(1.0, color='red', ls=':', lw=1.5, label='ratio=1')
    ax.set_xticks(x)
    ax.set_xticklabels(keys_present, rotation=45, ha='right', fontsize=8)
    ax.set_ylabel('NNT ratio (inter/intra)')
    ax.set_title('NNT Ratio — Moyenne vs VLAD')
    ax.legend(fontsize=8)

    # ── τ cross/intra ──────────────────────────────────────────────────────────
    ax = axes[1, 1]
    x_cross = x - 0.18
    x_intra = x + 0.18
    ax.bar(x_cross, [_vlad_tau_results[k]['cross_macro'] for k in keys_present],
           0.35, color=[COLORS[k] for k in keys_present], label='τ cross-image', alpha=0.9)
    ax.bar(x_intra, [_vlad_tau_results[k]['intra_macro'] for k in keys_present],
           0.35, color=[COLORS[k] for k in keys_present], label='τ intra-image', alpha=0.4)
    ax.set_xticks(x)
    ax.set_xticklabels(keys_present, rotation=45, ha='right', fontsize=8)
    ax.set_ylabel('τ (similarité cosine moyenne)')
    ax.set_title('τ cross vs intra — Moyenne vs VLAD')
    ax.legend(fontsize=8)

    plt.suptitle(
        'Analyse comparative — Moyenne vs VLAD\n'
        'block_0 et block_1 · K=16 et K=32 · PCA-50d · '
        'Catégories : ' + ', '.join(CATEGORIES[c] for c in CATS_VALID),
        fontsize=11,
    )
    plt.tight_layout()
    fig_path = OUT_DIR / 'vlad_comparison.png'
    plt.savefig(fig_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'→ Figure sauvegardée : {fig_path}')